# Denoising Transient Renders with Statistical Denoiser

This notebook demonstrates how to denoise transient (time-resolved) rendering using our statistical denoiser and compares it with other state-of-the-art methods.


## Setup

First, we import all necessary libraries and configure Mitsuba. For GPU acceleration, use the `cuda_ad_rgb` variant. For CPU-only systems, use `llvm_ad_rgb`.

In [1]:
import mitsuba as mi

mi.set_variant("llvm_ad_rgb")
import gc

import drjit as dr
import mitransient as mitr
import numpy as np
import torch

from denoisers.denoiserOptixTransient import denoiseOptixTransientTemporal
from denoisers.statTransientDenoiser import StatDenoiser


[mitsuba] Warning: Couldn't import the ipywidgets package. Installing this package is required for the system to properly log messages and print in Jupyter notebooks!


## Render Noisy Transient Data
We render the target scene at low sample count (SPP) to obtain noisy transient data. We also render G-buffers (albedo and normals) which will be used as auxiliary features for denoising.

In [2]:
SPP = 1024

steady_scene = mi.load_file("./scenes/steady/kitchen/scene.xml")
transient_scene = mi.load_file("./scenes/transient/kitchen/scene.xml")
sensor = steady_scene.sensors()[0]

mi.render(steady_scene, spp=1)
bitmap = sensor.film().bitmap()
res = dict(bitmap.split())
albedo = res["albedo"]
normals = res["nn"]

_, noisy, stats = mi.render(transient_scene, spp=SPP)

We convert the Dr.Jit variables to NumPy arrays and clear GPU memory to prepare for PyTorch-based denoising. This prevents memory overflow when working with large transient data.

In [3]:
estimands = stats[0]
estimands_variance = stats[1]
del stats
estimands_np = np.array(estimands, copy=True)
del estimands
estimands_variance_np = np.array(estimands_variance, copy=True)
del estimands_variance
noisy_np = np.array(noisy, copy=True)
del noisy
albedo_np = np.array(albedo)
del albedo
normals_np = np.array(normals)
del normals
del transient_scene, steady_scene, sensor, bitmap, res
dr.sync_thread()
dr.flush_malloc_cache()
torch.cuda.empty_cache()
torch.cuda.synchronize()
gc.collect()

68

## Denoise with Our Statistical Denoiser

We create and apply our statistical denoiser. The key parameters control the kernel size and the confidence interval:

- **spatial_radius**: Defines the radius for the spatial dimensions of the kernels
- **temporal_radius**: Defines the radius for the temporal dimension of the kernels
- **alpha**:  Confidence interval parameter
- **spp**: The samples per pixel used for rendering


In [ ]:
# Create denoiser

device = "cpu" # torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

stat_denoiser = StatDenoiser(
    spatial_radius=10,
    temporal_radius=1,
    alpha=0.1,
    spp=SPP,
    device=device,
)


with torch.no_grad():
    ours = stat_denoiser(
        noisy_np,
        albedo_np,
        normals_np,
        estimands_np,
        estimands_variance_np,
    )
ours = np.asarray(ours)
del StatDenoiser
torch.cuda.empty_cache()
torch.cuda.synchronize()
gc.collect()

Using device: cpu


## Compare with Alternative Methods

We also denoise using NVIDIA OptiX as a baseline for comparison. Note that in the paper we compare against the non-temporal version of OptiX, as we observe naively treating transient as temporal worsened results.


In [ ]:
optix = denoiseOptixTransientTemporal(noisy_np, albedo_np, normals_np)



## Visualize Results

Display the denoised results as videos to qualitatively compare the different methods.


In [ ]:
mitr.vis.show_video(mitr.vis.tonemap_transient(noisy_np), 2)
mitr.vis.show_video(mitr.vis.tonemap_transient(ours), 2)
mitr.vis.show_video(mitr.vis.tonemap_transient(optix), 2)

## Render Reference Ground Truth

To quantitatively evaluate the denoising quality, we render a high-quality reference video with much higher sample count.

In [ ]:
transient_scene = mi.load_file("./scenes/transient/kitchen/scene.xml")
_, reference, _ = mi.render(transient_scene, spp=15000)

reference_np = np.array(reference, copy=True)

del transient_scene, reference
dr.sync_thread()
dr.flush_malloc_cache()
torch.cuda.empty_cache()
torch.cuda.synchronize()
gc.collect()



4800

## Quantitative Evaluation

Finally, we compare the denoised results against the reference using three metrics:

- **RMSE**: Root Mean Squared Error
- **SSIM**: Structural Similarity Index
- **FLIP**: A Difference Evaluator for Alternating Images

In [ ]:
from metrics.metrics_functions import compute_ssim, compute_rmse, compute_flip


print("---------RMSE----------")
print("Ours: ", compute_rmse(ours, reference_np))
print("Optix: ", compute_rmse(optix, reference_np))
print("Noisy: ", compute_rmse(noisy_np, reference_np))

print("---------SSIM----------")
print("Ours: ", compute_ssim(ours, reference_np))
print("Optix: ", compute_ssim(optix, reference_np))
print("Noisy: ", compute_ssim(noisy_np, reference_np))

print("---------FLIP----------")
print("Ours: ", compute_flip(ours, reference_np))
print("Optix: ", compute_flip(optix, reference_np))
print("Noisy: ", compute_flip(noisy_np, reference_np))






---------RMSE----------
Ours:  0.0012884319
Optix:  0.0013865324
Noisy:  0.0025414613
---------SSIM----------
Ours:  0.99895257
Optix:  0.9985063
Noisy:  0.9962229
---------FLIP----------
Ours:  0.3691304686665535
Optix:  0.5076153284311294
Noisy:  0.4673364433646202
